# 👑 Supervisor Multi-Agent Architecture with LangGraph

Yeh notebook ek **Hierarchical Multi-Agent System** implement karta hai jahan ek central **Supervisor Agent** alag-alag specialized worker agents (Researcher, Analyst, Writer) ko coordinate aur delegate karta hai.

### 🔄 Workflow Architecture:
- **Supervisor**: Task ko analyze karta hai aur next best agent choose karta hai (ya finish hone par `FINISH` declare karta hai).
- **Researcher**: Data aur relevant information gather karta hai.
- **Analyst**: Gathered data ko analyze karke key insights extract karta hai.
- **Writer**: Final comprehensive report draft karta hai.
- **MemorySaver**: Stateful thread execution aur session checkpointing enable karta hai.

### 📦 1. Imports & Core Dependencies
- **Purpose**: LangGraph, LangChain messages, memory checkpointing, aur type hinting primitives import karna.
- **Key Modules**:
  - `MessagesState`: LangGraph built-in state jo conversation messages list ko automatically manage karta hai.
  - `MemorySaver`: In-memory checkpoint saver jo session state maintain karta hai.
  - `StateGraph`, `START`, `END`: Graph nodes aur execution flow define karne ke liye.

In [1]:
import os
from typing import Dict, Any, List, Literal
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()
print("Environment loaded successfully.")

### ⚙️ 2. Dynamic Model Initialization
- **Purpose**: Groq provider ke zariye `openai/gpt-oss-20b` LLM initialize karna.
- **Parameters**:
  - `model`: `openai/gpt-oss-20b`
  - `model_provider`: `groq`
  - `temperature`: `0.2` (deterministic routing aur structured output ke liye).

In [2]:
llm = init_chat_model(model="openai/gpt-oss-20b", model_provider="groq", temperature=0.2)
print("LLM initialized successfully.")

### 📋 3. Supervisor State Schema (`SupervisorState`)
- **Class**: `SupervisorState` (inherits from `MessagesState`)
- **Purpose**: Multi-agent graph execution ke dauran shared state aur agent outputs ko track karna.
- **Attributes**:
  - `messages`: Conversation history (`MessagesState` se auto-managed).
  - `next_agent`: Supervisor dwara chuna gaya agla worker (`researcher`, `analyst`, `writer`, ya `FINISH`).
  - `research_data`: Researcher agent ka raw findings output.
  - `analysis_data`: Analyst agent ka synthesized insights output.
  - `final_report`: Writer agent ka complete output.
  - `iteration_count`: Infinite loop prevention counter.

In [3]:
class SupervisorState(MessagesState):
    next_agent: str = ""
    research_data: str = ""
    analysis_data: str = ""
    final_report: str = ""
    iteration_count: int = 0
    max_iterations: int = 6

### 🧑‍💼 4. Supervisor Node & Decision Logic
- **Function**: `supervisor_node(state: SupervisorState) -> dict`
- **Purpose**: Current state inspect karna aur agla best worker agent decide karna.
- **Logic**:
  - Agar iteration limit cross ho gayi ya report ban chuki hai ➔ `FINISH`
  - Agar research missing hai ➔ `researcher`
  - Agar analysis missing hai ➔ `analyst`
  - Agar report missing hai ➔ `writer`
  - Sab complete ho chuka hai ➔ `FINISH`

In [4]:
def supervisor_node(state: SupervisorState) -> dict:
    count = state.get("iteration_count", 0) + 1
    if count > state.get("max_iterations", 6):
        return {"next_agent": "finish", "iteration_count": count}
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a project supervisor managing 3 workers:
1. researcher: Gathers background information and factual points.
2. analyst: Evaluates the gathered research and extracts structured insights.
3. writer: Produces the final polished summary report.

Current Status:
- Has Research: {has_research}
- Has Analysis: {has_analysis}
- Has Final Report: {has_report}

Rules:
- If research is missing, output: researcher
- If research exists but analysis is missing, output: analyst
- If analysis exists but final report is missing, output: writer
- If all steps are completed, output: finish
Output ONLY ONE WORD: researcher, analyst, writer, or finish."""),
        ("human", "Task: {task}")
    ])
    
    chain = prompt | llm
    user_task = state["messages"][0].content if state.get("messages") else "Research topic"
    decision = chain.invoke({
        "has_research": bool(state.get("research_data")),
        "has_analysis": bool(state.get("analysis_data")),
        "has_report": bool(state.get("final_report")),
        "task": user_task
    }).content.strip().lower()
    
    valid_choices = {"researcher", "analyst", "writer", "finish"}
    clean_decision = decision if decision in valid_choices else "finish"
    return {"next_agent": clean_decision, "iteration_count": count}

### 🔬 5. Worker Nodes (Researcher, Analyst, Writer)
- **Researcher Node**: User task ke hisaab se factual points aur raw findings extract karta hai.
- **Analyst Node**: Research points ko categorize karke pros, cons, aur core insights nikalta hai.
- **Writer Node**: Findings aur analysis ko ek structured executive summary mein compile karta hai.

In [5]:
def researcher_node(state: SupervisorState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert researcher. Gather 3-4 key factual points on the given topic."),
        ("human", "{task}")
    ])
    res = (prompt | llm).invoke({"task": state["messages"][0].content})
    return {
        "research_data": res.content,
        "messages": [AIMessage(content=f"[Researcher]: Research gathered:\n{res.content}")]
    }

def analyst_node(state: SupervisorState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an analyst. Analyze the provided research findings and extract key technical and business implications."),
        ("human", "Research:\n{research}")
    ])
    res = (prompt | llm).invoke({"research": state.get("research_data", "")})
    return {
        "analysis_data": res.content,
        "messages": [AIMessage(content=f"[Analyst]: Analysis completed:\n{res.content}")]
    }

def writer_node(state: SupervisorState) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an executive writer. Synthesize the research and analysis into a clean executive summary with key takeaways."),
        ("human", "Research:\n{research}\n\nAnalysis:\n{analysis}")
    ])
    res = (prompt | llm).invoke({
        "research": state.get("research_data", ""),
        "analysis": state.get("analysis_data", "")
    })
    return {
        "final_report": res.content,
        "messages": [AIMessage(content=f"[Writer]: Executive Report:\n{res.content}")]
    }

### 🔀 6. Supervisor Conditional Edge & Graph Construction
- **Routing Function**: `route_next_agent(state: SupervisorState) -> str`
- **Purpose**: Supervisor ke decision (`next_agent`) ke mutabiq agle worker node par branch karna ya execution `END` karna.
- **Checkpointing**: `MemorySaver()` use karke StateGraph ko compile karna.

In [6]:
def route_next_agent(state: SupervisorState) -> str:
    next_step = state.get("next_agent", "finish").lower()
    if next_step == "researcher":
        return "researcher"
    elif next_step == "analyst":
        return "analyst"
    elif next_step == "writer":
        return "writer"
    return END

builder = StateGraph(SupervisorState)

builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("analyst", analyst_node)
builder.add_node("writer", writer_node)

builder.add_edge(START, "supervisor")

builder.add_conditional_edges(
    "supervisor",
    route_next_agent,
    {
        "researcher": "researcher",
        "analyst": "analyst",
        "writer": "writer",
        END: END
    }
)

builder.add_edge("researcher", "supervisor")
builder.add_edge("analyst", "supervisor")
builder.add_edge("writer", "supervisor")

checkpointer = MemorySaver()
supervisor_graph = builder.compile(checkpointer=checkpointer)
print("Supervisor Multi-Agent Graph compiled successfully!")

### 🚀 7. Multi-Agent System Execution with Thread ID
- **Purpose**: Multi-agent graph ko live topic ke sath invoke karna aur stream updates observe karna.
- **Thread ID**: `supervisor_run_1` session state ko persist karta hai.

In [7]:
config = {"configurable": {"thread_id": "supervisor_run_1"}}

initial_input = {
    "messages": [HumanMessage(content="Evaluate the impact of Small Language Models on edge AI devices.")]
}

print("Starting Supervisor Multi-Agent Workflow...\n")
for event in supervisor_graph.stream(initial_input, config=config, stream_mode="updates"):
    for node_name, state_update in event.items():
        print(f"🔹 [Node: {node_name}] completed.")
        if "next_agent" in state_update:
            print(f"   Supervisor routed to: {state_update['next_agent']}")

final_state = supervisor_graph.get_state(config)
print("\n=================== FINAL EXECUTIVE REPORT ===================")
print(final_state.values.get("final_report", "Task Completed"))